<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Meeting transcripts with speakers

**[Meeting transcripts with speakers](https://learning.nextia-ai.com/courses/speech/m10/meeting-transcripts-with-speakers/)** · Speech AI: Transcribe, Understand and Speak · Case study 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** diarize and transcribe real recorded meetings, measure who-spoke-when (DER, with its missed, false and confusion parts) and the words (WER), score whole notes with names (cpWER), compare a far microphone with one headset per person, and turn a transcript into action items.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in a few minutes |
| **Needs** | An internet connection for the setup cells (about 27 MB of data, a 27 MB speaker model and the 75 MB Whisper tiny model). No account and no key: the LLM's answers are recorded. |
| **You should know** | DER and channels from [Split by channel first](https://learning.nextia-ai.com/courses/speech/m05/split-by-channel-first/); embeddings and clustering from [Speaker embeddings and clustering](https://learning.nextia-ai.com/courses/speech/m05/speaker-embeddings-and-clustering/); WER from [Word error rate, step by step](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0, matplotlib 3.11.2, scikit-learn 1.9.1, jiwer 4.0.0, faster-whisper 1.2.1 and sherpa-onnx 2.0.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m10/meeting-transcripts-with-speakers/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M10-L02-meeting-transcripts-with-speakers/meeting-transcripts-with-speakers-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, scikit-learn 1.9.1, jiwer 4.0.0, faster-whisper 1.2.1, sherpa-onnx 2.0.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. faster-whisper brings its own engine (CTranslate2) and the Silero voice detector: the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 scikit-learn==1.9.1 jiwer==4.0.0 faster-whisper==1.2.1 sherpa-onnx==2.0.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'scikit-learn', 'jiwer', 'faster-whisper', 'sherpa-onnx']:
    print(name, version(name))

## Setup: get the meetings

Run the next cell. It downloads the files of this case study (about 27 MB) into a folder `ami-meetings`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ami-meetings`, then run the cell again.

What you get: a 5-minute excerpt of AMI meeting IS1008a (the four headsets and the far microphone, 16 kHz FLAC), the word-level reference of four AMI meetings, and the **recorded** outputs of the full meetings (Whisper transcripts, the turns of every diarization method, and chat-small's action items). The full recordings are about 150 MB a meeting, so the notebook scores the recorded outputs and runs the live pipeline on the excerpt.

Source: [AMI Meeting Corpus](https://groups.inf.ed.ac.uk/ami/corpus/), University of Edinburgh and the AMI Consortium, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Nextia Learning cut the excerpt and converted it to FLAC, and extracted the word reference from the AMI manual annotations 1.6.2. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M10-L02-meeting-transcripts-with-speakers/dataset.md) has the details.

In [ ]:
# Setup: download the files into ami-meetings/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
HERE_FOLDER = "M10-L02-meeting-transcripts-with-speakers/"
FILES = {
    "data/audio/IS1008a-excerpt.Array1-01.flac": "09e788a609dddd0109214507e7d3a1bc951c9b0a8a3b6c3436e18bef4bd5c598",
    "data/audio/IS1008a-excerpt.Headset-0.flac": "1037dda02002ec1a9f7290275e479497b1810ecac52c4b126e4dd2035acd4ddf",
    "data/audio/IS1008a-excerpt.Headset-1.flac": "f2ce9654ae400984a2e231f63e4c5bb77f611ce80a516e50f512d79e6c2d9c58",
    "data/audio/IS1008a-excerpt.Headset-2.flac": "eb49ac5fd41efedb604ca913d45409e8a213835f583997e7a477403758014732",
    "data/audio/IS1008a-excerpt.Headset-3.flac": "3609b593eef84d1604aa7d1ce0dd3e1df1ce41c6a92c62315954a936422cdd39",
    "data/recorded/action-items.jsonl": "ea078c445675df12261d312a027038b8b498b2de2b1df4f8f42d913acebc2a84",
    "data/recorded/diarization.json": "ccbd946c6d6b76818b036f7996b68369b1a5db1a44eea61ded011526355484ca",
    "data/recorded/whisper-base.jsonl": "39f854d200b898121a8353c3170e4ca7335125a596c8cfbf904208435192700f",
    "data/recorded/whisper-small-vad.jsonl": "36ff6578cfcf00dec543e41be600558c583e9ab7e682dfd988b9273b3b053d13",
    "data/recorded/whisper-small.jsonl": "9558629e6a2ed0acfe60177ffa65f5ef8f76fc83fdaca1656f0e9645cb0fc5b2",
    "data/recorded/whisper-tiny.jsonl": "dc3ae3ca85d00efe1a5176ac4bfe179e3531576ed375c68a5d9c5535dafc2d55",
    "data/reference.json": "3752b0f18fc4d934e5ead6db6780e3e0d17e74186250a66b728754fce02782e1",
}
FILES["compare.py"] = "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b"
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ami-meetings").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + (name if name == "compare.py" else HERE_FOLDER + name), path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ami-meetings and run again.")
os.chdir(PROJECT)
print(f"Ready: {len(FILES)} files in ami-meetings.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: the meeting tools

The next cell defines the tools of this case study. They are the course's tools, made ready for meetings:

- `der(found, truth, seconds)`: the diarization error rate of [Split by channel first](https://learning.nextia-ai.com/courses/speech/m05/split-by-channel-first/), now with **overlap**: one moment can have two true speakers and two found speakers. It returns the missed, false and confusion parts apart, and the best matching of names.
- `silero(x)`: the Silero voice activity detector (the copy inside faster-whisper), as in the project.
- `by_headsets(heads, names, mode)`: who spoke when from one headset per person: `"alone"` (each headset's speech is its wearer's) or `"louder"` (a moment that several headsets call speech goes to the loudest one).
- `norm(text)`: the course's normalisation guide, after three meeting rules (letters said one by one, more fillers, short forms).
- `cpwer(ref_words, hyp_words)`: the word error rate with names, explained in section 4.

Run the cell. It prints the meeting rules.

In [ ]:
import json, re, time
import numpy as np
import soundfile as sf
from scipy.optimize import linear_sum_assignment
STEP = 0.01  # 10 ms frames

MEETING_RULES = """Meeting rules (before the normalisation guide, for the reference and the transcript alike)
1. Letters said one by one are one word: AMI writes "L_C_D_", Whisper writes "LCD" -> "lcd".
2. More sounds are fillers: mm-hmm, mhm, uh-huh, hmm, mm, ah-ha, oh (alone).
3. Short forms become the full word: ok, 'kay -> okay; 'cause -> because; gonna -> going to; wanna -> want to."""
SHORT = {"ok": "okay", "'kay": "okay", "kay": "okay", "'cause": "because", "cos": "because", "gonna": "going to",
         "wanna": "want to", "'til": "until", "'em": "them"}
MORE_FILLERS = {"mm-hmm", "mhm", "uh-huh", "hmm", "mm", "ah-ha", "oh", "huh", "uh-oh"}

def meeting_rules(text):
    text = re.sub(r"\b((?:[A-Za-z]_){2,})", lambda m: m.group(1).replace("_", ""), text)
    text = re.sub(r"\b([A-Z])\.(?=[A-Z]\.)", r"\1", text).replace("_", "")
    out = []
    for t in text.split():
        k = re.sub(r"[.,!?;:\"]+$", "", t.lower()).strip('"')
        if k not in MORE_FILLERS:
            out.append(SHORT.get(k, t))
    return " ".join(out)

def norm(text, rules=True):
    return cp.normalise(meeting_rules(text) if rules else text, "full", "en")

def frames_of(turns, names, n):
    m = np.zeros((len(names), n), bool)
    for s, e, w in turns:
        if w in names:
            m[names.index(w), max(0, int(round(s / STEP))):max(0, int(round(e / STEP)))] = True
    return m

def turns_from_frames(active, name, min_len=0.1):
    out, start = [], None
    for i, a in enumerate(list(active) + [False]):
        if a and start is None:
            start = i
        elif not a and start is not None:
            if (i - start) * STEP >= min_len:
                out.append((round(start * STEP, 2), round(i * STEP, 2), name))
            start = None
    return out

def der(found, truth, seconds, collar=0.25):
    """DER with overlap: missed + false + confusion, over the true speech time; a collar of 0.25 s is not scored."""
    n = int(seconds / STEP) + 1
    tn, fn = sorted({t[2] for t in truth}), sorted({f[2] for f in found})
    T, P = frames_of(truth, tn, n), frames_of(found, fn, n)
    score = np.ones(n, bool)
    for s, e, _ in truth:
        for b in (s, e):
            score[max(0, int((b - collar) / STEP)):int((b + collar) / STEP)] = False
    T, P = T[:, score], P[:, score]
    nref, nsys = T.sum(0), P.sum(0)
    correct, mapping = 0, {}
    if fn and tn:
        overlap = (P[:, None, :] & T[None, :, :]).sum(2)
        rows, cols = linear_sum_assignment(-overlap)          # the best one-to-one matching of names
        correct = int(sum(overlap[r, c] for r, c in zip(rows, cols)))
        mapping = {fn[r]: tn[c] for r, c in zip(rows, cols)}
    total = max(int(nref.sum()), 1)
    missed = int(np.maximum(nref - nsys, 0).sum())
    false = int(np.maximum(nsys - nref, 0).sum())
    confusion = int(np.minimum(nref, nsys).sum()) - correct
    return {"der": (missed + false + confusion) / total, "missed": missed / total, "false": false / total,
            "confusion": confusion / total, "mapping": mapping, "speech": total}

def silero(x, sr=16000, threshold=0.5):
    from faster_whisper.vad import VadOptions, get_speech_timestamps
    opts = VadOptions(threshold=threshold, min_silence_duration_ms=300, speech_pad_ms=100)
    return [(t["start"] / sr, t["end"] / sr) for t in get_speech_timestamps(x, opts, sampling_rate=sr)]

def frame_db(x, sr=16000):
    hop = int(sr * STEP)
    k = len(x) // hop
    return 10 * np.log10((x[: k * hop].reshape(k, hop) ** 2).mean(1) + 1e-12)

def by_headsets(heads, names, mode="louder", threshold=0.5, margin_db=0.0):
    """One headset per person. "alone": each headset's speech is its wearer's. "louder": a moment that several
    headsets call speech goes to the loudest headset. "matched": loudness against each headset's own speech level,
    and every headset within margin_db of the loudest keeps the moment."""
    n = min(len(h) for h in heads) // int(16000 * STEP)
    active = np.zeros((len(heads), n), bool)
    for k, h in enumerate(heads):
        for s, e in silero(h, 16000, threshold):
            active[k, int(s / STEP):int(e / STEP)] = True
    if mode != "alone":
        energy = np.stack([frame_db(h)[:n] for h in heads])
        if mode == "matched":
            for k in range(len(heads)):
                own = energy[k][active[k]]
                energy[k] -= np.percentile(own, 90) if own.size else 0
        e = np.where(active, energy, -np.inf)
        keep = e >= e.max(0) - margin_db
        active = np.where(active.sum(0) >= 2, active & keep, active)
    turns = []
    for k, name in enumerate(names):
        turns += turns_from_frames(active[k], name)
    return sorted(turns)

def speaker_at(turns, t, reach=0.5):
    """The speaker of the turn that holds time t; a word in a short gap goes to the nearest turn within reach."""
    for s, e, w in turns:
        if s <= t <= e:
            return w
    near = [(min(abs(t - s), abs(t - e)), w) for s, e, w in turns]
    best = min(near, default=(reach + 1, "?"))
    return best[1] if best[0] <= reach else "?"

def errors(ref, hyp):
    """Word edits between two normalised texts (substitutions + deletions + insertions)."""
    import jiwer
    if not ref.split() or not hyp.split():
        return len(ref.split()) + len(hyp.split())
    o = jiwer.process_words(ref, hyp)
    return o.substitutions + o.deletions + o.insertions

def by_speaker(words):
    out = {}
    for s, e, w, t in sorted(words):
        out.setdefault(w, []).append(t)
    return {k: norm(" ".join(v)) for k, v in out.items()}

def cpwer(ref_words, hyp_words):
    """Join each speaker's words; match found speakers to true speakers with the fewest errors; count all errors."""
    Rs, Hs = by_speaker(ref_words), by_speaker(hyp_words)
    rk, hk = sorted(Rs), sorted(Hs)
    size = max(len(rk), len(hk))
    cost = np.zeros((size, size))
    for i in range(size):
        for j in range(size):
            cost[i, j] = errors(Rs[rk[i]] if i < len(rk) else "", Hs[hk[j]] if j < len(hk) else "")
    rows, cols = linear_sum_assignment(cost)
    n = sum(len(v.split()) for v in Rs.values())
    return {"cpwer": cost[rows, cols].sum() / max(n, 1), "errors": int(cost[rows, cols].sum()), "ref_words": n}

print(MEETING_RULES)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data, briefly

Four AMI meetings: three for the comparison, and IS1008a kept apart as **new input** for the end-to-end run. Each meeting has four people (A to D) with a role: project manager (PM), marketing expert (ME), user interface designer (UI) and industrial designer (ID). Each person wore a headset; a microphone on the table (Array1-01) heard the whole room.

Run the next cell. It prints each meeting's length, words, turns and people.

In [ ]:
ref = json.load(open("data/reference.json"))
excerpt = ref.pop("_excerpt")
compare_meetings = [m for m in ref if ref[m]["set"] == "compare"]
for mid, m in ref.items():
    people = ", ".join(f"{k}={s['role']} {s['sex']} {s['native_language']}" for k, s in sorted(m["speakers"].items()))
    print(f"{mid} ({m['set']}): {m['duration'] / 60:.1f} min, {len(m['words'])} words, {len(m['turns'])} turns; {people}")

In a meeting, people talk at the same time. The next cell measures the **overlap**: the share of the speech time with two or more voices.

In [ ]:
def overlap_share(mid):
    """The share of the speech time in which two or more people speak at once."""
    m = ref[mid]
    n = int(m["duration"] / STEP) + 1
    count = frames_of([tuple(t) for t in m["turns"]], sorted(m["speakers"]), n).sum(0)
    return float((count >= 2).sum() / (count >= 1).sum())

for mid in ref:
    print(mid, f"overlap {overlap_share(mid):.1%}")

> **Check.** ES2004a 15.8%, IS1009a 13.6%, IS1001a 9.4%. Larkfield's calls almost never overlap; meetings do, often for short "yeah" and "mm-hmm".

## 2. DER with overlap, on a tiny example

A method that gives each moment to **one** speaker cannot be right during overlap. Truth: A speaks from 0 to 4 s, B from 3 to 5 s. Found: speaker s1 from 0 to 3.5 s, s2 from 3.5 to 5 s.

> **Predict.** Which parts of the DER will you see: missed, false, confusion?

In [ ]:
truth = [(0, 4, "A"), (3, 5, "B")]          # B starts while A is still talking: 1 s of overlap
found = [(0, 3.5, "s1"), (3.5, 5, "s2")]     # the system gives every moment to one speaker
r = der(found, truth, 5, collar=0)
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in r.items()})

> **Check.** The best matching is s1 → A and s2 → B. True speech time: 4 + 2 = 6 s. From 3 to 4 s two people speak, but only one is found: 1 s of 6 is missed. Everything that was found is under a right name, so confusion is 0. Missed 0.17, confusion 0.0, DER 0.17.

## 3. Score every diarization method (recorded on the full meetings)

The turns of six methods were recorded on the three full meetings. Run the cell: it scores them with `der` and pools the three meetings.

In [ ]:
diar = json.load(open("data/recorded/diarization.json"))
METHODS = ["far_one", "far_emb", "far_emb_3s", "mix_emb", "ihm_alone", "ihm_louder"]

def pooled_der(method, meetings):
    """DER parts over several meetings: each part weighted by the meeting's scored speech time."""
    tot = {"der": 0, "missed": 0, "false": 0, "confusion": 0, "speech": 0}
    for mid in meetings:
        m = ref[mid]
        truth = [tuple(t) for t in m["turns"]]
        r = der([tuple(t) for t in diar[mid][method]["turns"]], truth, m["duration"])
        speech = r["speech"]  # weight: the scored true speech time of this meeting
        for k in ("der", "missed", "false", "confusion"):
            tot[k] += r[k] * speech
        tot["speech"] += speech
    return {k: tot[k] / tot["speech"] for k in ("der", "missed", "false", "confusion")}

table = {method: pooled_der(method, compare_meetings) for method in METHODS}
for method, r in table.items():
    print(f"{method:11s} DER {r['der']:6.1%}  missed {r['missed']:5.1%}  false {r['false']:6.1%}  confusion {r['confusion']:5.1%}")

> **Check.** The baseline (far microphone, one speaker): DER 48.2%. The far microphone with embeddings: 50.3%, **not better**. Headsets, each alone: 145.3% (false speech 131.6%: every headset hears its neighbours). Headsets, louder wins: 26.9%, with confusion 1.5%; most of what is left is missed speech (24.4%).

## 4. Why the embeddings fail on the far microphone

The next two cells download the speaker model of Module 5 and compare 1.5-second pieces of the excerpt: pieces from the **same** person and pieces from **different** people, on the far microphone and on the four headsets added together.

> **Predict.** On the far microphone, is "same person" clearly more alike than "different people"?

In [ ]:
# The speaker model: WeSpeaker ResNet-34, trained on VoxCeleb (Apache-2.0), the sherpa-onnx copy (27 MB).
import sherpa_onnx
# Two places for the same file (the checksum is checked either way): GitHub, then a copy on Hugging Face.
MODEL_URLS = ["https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-recongition-models/"
              "wespeaker_en_voxceleb_resnet34_LM.onnx",
              "https://huggingface.co/csukuangfj/speaker-embedding-models/resolve/main/"
              "wespeaker_en_voxceleb_resnet34_LM.onnx"]
MODEL_SHA256 = "e9848563da86f263117134dfd7ad63c92355b37de492b55e325400c9d9c39012"
model_file = Path("models/wespeaker_en_voxceleb_resnet34_LM.onnx")
if not (model_file.exists() and sha256(model_file) == MODEL_SHA256):
    model_file.parent.mkdir(exist_ok=True)
    for url in MODEL_URLS:
        try:
            download(url, model_file)
        except SystemExit:
            continue  # this server did not answer: try the next one
        if model_file.exists() and sha256(model_file) == MODEL_SHA256:
            break
    else:
        raise SystemExit("Could not download the speaker model with the right checksum. Delete the folder models "
                         "and run again later.")
extractor = sherpa_onnx.SpeakerEmbeddingExtractor(
    sherpa_onnx.SpeakerEmbeddingExtractorConfig(model=str(model_file), num_threads=2))

def embed(x, sr=16000):
    stream = extractor.create_stream()
    stream.accept_waveform(sr, x)
    stream.input_finished()
    e = np.array(extractor.compute(stream))
    return e / np.linalg.norm(e)

def level(x, target=-25.0):
    """Set the loudness to target dBFS (the far microphone is quiet)."""
    return np.clip(x * 10 ** ((target - 20 * np.log10(np.sqrt(np.mean(x ** 2)) + 1e-12)) / 20), -1, 1).astype(np.float32)

print("Speaker model ready:", extractor.dim, "numbers per embedding.")

In [ ]:
t0 = excerpt["start"]
ex = ref[excerpt["meeting"]]
names = sorted(ex["speakers"], key=lambda k: ex["speakers"][k]["channel"])
heads = [sf.read(f"data/audio/{excerpt['meeting']}-excerpt.Headset-{ex['speakers'][k]['channel']}.flac", dtype="float32")[0]
         for k in names]
far = sf.read(f"data/audio/{excerpt['meeting']}-excerpt.Array1-01.flac", dtype="float32")[0]
ex_truth = [(round(s - t0, 2), round(e - t0, 2), w) for s, e, w in ex["turns"] if s >= t0 and e <= excerpt["end"]]
ex_seconds = len(far) / 16000

def alike(x):
    """Mean cosine of 1.5 s pieces inside one person's turn: same person, and different people."""
    pieces = [(s + i * 1.5, s + (i + 1) * 1.5, w) for s, e, w in ex_truth for i in range(int((e - s) // 1.5))]
    E = np.stack([embed(x[int(a * 16000):int(b * 16000)]) for a, b, _ in pieces])
    who = np.array([w for *_, w in pieces])
    sim = E @ E.T
    iu = np.triu_indices(len(pieces), 1)
    same = who[iu[0]] == who[iu[1]]
    return round(float(sim[iu][same].mean()), 2), round(float(sim[iu][~same].mean()), 2), len(pieces)

alike_far = alike(level(far))
alike_mix = alike(level(sum(heads)))
print("far microphone:  same person", alike_far[0], " different people", alike_far[1], f"({alike_far[2]} pieces)")
print("headset mix:     same person", alike_mix[0], " different people", alike_mix[1])

> **Check.** Far microphone: same 0.86, different 0.8. Headset mix: same 0.68, different 0.54. On this short excerpt the far microphone keeps a small gap; on the three full meetings (the page) the gap is almost zero. The model hears the room more than the person, so the clustering cannot split the voices.

## 5. Words with names: cpWER

WER ignores names; DER ignores words. Notes need both. **cpWER** (concatenated minimum-permutation WER) joins each person's words, matches each found speaker to a true speaker in the way that gives the fewest errors, then counts every error, including words under the wrong name.

In [ ]:
truth_words = [(0, 1, "A", "we need a budget"), (2, 3, "B", "I will write it")]
found_words = [(0, 1, "2", "we need a budget"), (2, 3, "1", "I will right it")]
print(cpwer(truth_words, found_words))

> **Check.** One error ("right" for "write") in 8 words: cpWER 0.125.

Now the four end-to-end setups, with Whisper small, on the three full meetings (recorded transcripts):

In [ ]:
small = {(r["meeting"], r["stream"]): r for r in map(json.loads, open("data/recorded/whisper-small.jsonl"))}

def ref_words(mid):
    return [(w["start"], w["end"], w["speaker"], w["text"]) for w in ref[mid]["words"]]

def inside(turns, t, reach=0.2):
    return any(s - reach <= t <= e + reach for s, e, _ in turns)

def setups(rec, mid):
    """The words of each end-to-end setup, each with a speaker name."""
    m = ref[mid]
    far = rec[(mid, "Array1-01")]["words"]
    emb = [tuple(t) for t in diar[mid]["far_emb"]["turns"]]
    louder = [tuple(t) for t in diar[mid]["ihm_louder"]["turns"]]
    alone, kept = [], []
    for k, s in m["speakers"].items():
        for a, b, text in rec[(mid, f"Headset-{s['channel']}")]["words"]:
            alone.append((a, b, k, text))
            if inside([t for t in louder if t[2] == k], (a + b) / 2):
                kept.append((a, b, k, text))
    return {"far_one": [(a, b, "S1", t) for a, b, t in far],
            "far_emb": [(a, b, speaker_at(emb, (a + b) / 2), t) for a, b, t in far],
            "ihm_alone": alone, "ihm_louder": kept}

def pooled_cpwer(rec, meetings):
    tot = {}
    for mid in meetings:
        for key, words in setups(rec, mid).items():
            r = cpwer(ref_words(mid), words)
            e, n = tot.get(key, (0, 0))
            tot[key] = (e + r["errors"], n + r["ref_words"])
    return {k: e / n for k, (e, n) in tot.items()}

cp_small = pooled_cpwer(small, compare_meetings)
for key, v in cp_small.items():
    print(f"{key:11s} cpWER {v:.1%}")

> **Check.** Far mic, one speaker 89.3%; far mic with embeddings 86.9%; headsets alone 152.8%; headsets with louder wins 26.5%.

## 6. Predict, then run: Whisper size and microphone

> **Predict.** Which helps more: a bigger Whisper on the far microphone, or a headset with Whisper tiny?

In [ ]:
def wer_far_and_headsets(rec):
    far_e = far_n = head_e = head_n = 0
    for mid in compare_meetings:
        r = norm(" ".join(t for *_, t in ref_words(mid)))
        far_e += errors(r, norm(" ".join(t for *_, t in rec[(mid, "Array1-01")]["words"])))
        far_n += len(r.split())
        words = setups(rec, mid)["ihm_louder"]
        for k in ref[mid]["speakers"]:
            rk = norm(" ".join(t for *_, w, t in ref_words(mid) if w == k))
            head_e += errors(rk, norm(" ".join(t for *_, w, t in words if w == k)))
            head_n += len(rk.split())
    return far_e / far_n, head_e / head_n

recorded = {size: {(r["meeting"], r["stream"]): r for r in map(json.loads, open(f"data/recorded/whisper-{size}.jsonl"))}
            for size in ("tiny", "base", "small")}
sizes = {}
for size, rec in recorded.items():
    far_wer, head_wer = wer_far_and_headsets(rec)
    rtf = sum(r["seconds"] for (mid, st), r in rec.items() if mid in compare_meetings and st != "Array1-01") / sum(
        ref[mid]["duration"] for mid in compare_meetings)
    sizes[size] = (far_wer, head_wer, rtf)
    print(f"Whisper {size:5s}  far mic WER {far_wer:.1%}   headsets WER {head_wer:.1%}   compute for the 4 headsets: {rtf:.3f} of the audio time")

> **Check.** Far microphone: tiny 43.6%, base 37.4%, small 30.7%. Headsets: tiny 30.1%, base 26.9%, small 26.5%. The microphone matters more than the model size.

## 7. Change one thing: louder wins, live on the new meeting

The excerpt is 5 minutes of IS1008a, the meeting we kept apart. Run who-spoke-when from the headsets, live, without and with "louder wins".

In [ ]:
for mode in ("alone", "louder"):
    t = time.perf_counter()
    turns = by_headsets(heads, names, mode=mode)
    r = der(turns, ex_truth, ex_seconds)
    print(f"{mode:7s} DER {r['der']:.1%}  missed {r['missed']:.1%}  false {r['false']:.1%}  confusion {r['confusion']:.1%}  ({time.perf_counter() - t:.1f} s)")
live_turns = by_headsets(heads, names, mode="louder")

> **Check.** Alone: DER 297.7% (false 293.5%). Louder wins: DER 29.0% (missed 22.4%). One rule removes nearly all of the crosstalk.

## 8. Failure case: a setting tuned in one room

In the first meeting we tried, a softer rule helped: compare each headset with its own speech level, and keep every headset within 10 dB of the loudest (to keep real overlap). Run it here.

In [ ]:
turns_margin = by_headsets(heads, names, mode="matched", margin_db=10)
r = der(turns_margin, ex_truth, ex_seconds)
print(f"matched, 10 dB margin: DER {r['der']:.1%}  missed {r['missed']:.1%}  false {r['false']:.1%}  confusion {r['confusion']:.1%}")
for mid in compare_meetings:
    print(mid, f"recorded on the full meeting: margin DER {der([tuple(t) for t in diar[mid]['ihm_margin']['turns']], [tuple(t) for t in ref[mid]['turns']], ref[mid]['duration'])['der']:.1%}")

> **Check.** On this excerpt: DER 167.5% (false 155.6%). It helped in the Edinburgh room and failed in the IDIAP rooms, where the headsets hear more of the room. **Fix:** choose settings on several meetings and rooms, and keep the plain rule when a smarter one is not better everywhere.

## 9. End to end on new input: notes with names

Now the recommended setup on the excerpt: louder wins, then Whisper on each headset, keeping only the wearer's own words. The cell downloads Whisper tiny (75 MB) once; the run takes about a minute on a laptop.

In [ ]:
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download
whisper = WhisperModel(snapshot_download("Systran/faster-whisper-tiny", revision="d90ca5fe260221311c53c58e660288d3deb8d356"),
                       device="cpu", compute_type="int8", cpu_threads=4)
t = time.perf_counter()
notes_words = []
for k, h in zip(names, heads):
    segments, _ = whisper.transcribe(h, language="en", beam_size=1, word_timestamps=True,
                                     condition_on_previous_text=False)
    mine = [x for x in live_turns if x[2] == k]
    for seg in segments:
        for w in seg.words or []:
            if inside(mine, (w.start + w.end) / 2):          # louder wins: keep the wearer's own words
                notes_words.append((w.start, w.end, k, w.word.strip()))
print(f"{len(notes_words)} words in {time.perf_counter() - t:.0f} s")

In [ ]:
def notes(words, roles):
    """Words with names -> lines of notes: a new line when the speaker changes or after a 2-second pause."""
    lines, cur = [], None
    for s, e, w, text in sorted(words):
        if cur and cur[1] == w and s - cur[3] < 2.0:
            cur[2].append(text); cur[3] = e
        else:
            if cur:
                lines.append(cur)
            cur = [s, w, [text], e]
    lines.append(cur)
    return [f"[{int((s + t0) // 60):02d}:{int((s + t0) % 60):02d}] {roles[w]}: {' '.join(ts)}" for s, w, ts, _ in lines]

roles = {k: ex["speakers"][k]["role"] for k in names}
lines = notes(notes_words, roles)
print("\n".join(lines[-12:]))
ex_ref = [(w["start"] - t0, w["end"] - t0, w["speaker"], w["text"]) for w in ex["words"] if t0 <= w["start"] < excerpt["end"]]
live = cpwer(ex_ref, notes_words)
print(f"\ncpWER of these notes (Whisper tiny, live): {live['cpwer']:.1%} on {live['ref_words']} words")

> **Check.** A cpWER of about 18% on 812 reference words (Whisper tiny, live; the last digit can change between runs). The page's recorded run uses Whisper small on the whole meeting.

The action items come from an LLM that read the transcript of the **whole** meeting (never the audio). Its answer was recorded once, so you need no key:

In [ ]:
actions = [json.loads(line) for line in open("data/recorded/action-items.jsonl")]
for row in actions:
    if row["meeting"] == excerpt["meeting"] and row["transcript"] == "headsets":
        print("Recorded answer from chat-small (Whisper small on the headsets, louder wins), US$", row["usd"])
        for item in row["answer"]["action_items"]:
            print("-", item["owner"], "|", item["task"], "| due:", item["due"])
print("\nThe AMI annotators' actions for this meeting:")
for a in ref[excerpt["meeting"]]["actions"]:
    print("-", a)

## 10. Each person, and the groups

Run the cell: the WER of each of the 12 people on their own headset (Whisper small, louder wins), worst first.

In [ ]:
def person_rows(rec):
    """(WER, words, person, role, sex, first language) for each person, on their own headset (louder wins)."""
    out = []
    for mid in compare_meetings:
        words = setups(rec, mid)["ihm_louder"]
        for k, s in ref[mid]["speakers"].items():
            rk = norm(" ".join(t for *_, w, t in ref_words(mid) if w == k))
            e = errors(rk, norm(" ".join(t for *_, w, t in words if w == k)))
            out.append((e / max(len(rk.split()), 1), len(rk.split()), f"{mid}.{k}", s["role"], s["sex"], s["native_language"]))
    return out

rows = person_rows(small)
for r in sorted(rows, reverse=True):
    print(f"{r[2]}  {r[3]}  {r[4]}  {r[5]:13s} words {r[1]:5d}  WER {r[0]:.1%}")

In [ ]:
def group_wer(test):
    e = n = 0
    for w, words, *_ in [r for r in rows if test(r)]:
        e += w * words; n += words
    return e / n
print("women", f"{group_wer(lambda r: r[4] == 'F'):.1%}", "  men", f"{group_wer(lambda r: r[4] == 'M'):.1%}")
print("native English", f"{group_wer(lambda r: r[5] == 'English'):.1%}", "  other first language", f"{group_wer(lambda r: r[5] != 'English'):.1%}")

### Change one thing: Whisper's own VAD

The recorded transcripts `whisper-small-vad` were made with `vad_filter=True`: Whisper skips what its Silero VAD calls silence. On the headsets, the WER of all three meetings falls (the page has the numbers).

> **Predict.** Is it better for every person?

In [ ]:
small_vad = {(r["meeting"], r["stream"]): r for r in map(json.loads, open("data/recorded/whisper-small-vad.jsonl"))}
with_vad = {r[2]: r[0] for r in person_rows(small_vad)}
better = sum(with_vad[r[2]] < r[0] for r in rows)
print(f"With Whisper's VAD: better for {better} of {len(rows)} people")
for r in sorted(rows, reverse=True)[:3]:
    print(f"{r[2]}  without VAD {r[0]:.1%}   with VAD {with_vad[r[2]]:.1%}")

> **Check.** Better for 10 of 12 people. The worst person goes from 89.4% to 161.7%: her headset kept words of her neighbours. A better average can hide a worse worst case.

> **Warning.** Twelve people in three meetings are far too few to say anything about women and men or first languages. Read these as a reason to check every person, not as a result about groups.

## 11. Your turn

> **Your turn.** (a) The DER of "headsets, each alone" on IS1009a: put its **false** part, rounded to 3 decimals, in `task_false`.

In [ ]:
m = ref["IS1009a"]
r = der([tuple(t) for t in diar["IS1009a"]["ihm_alone"]["turns"]], [tuple(t) for t in m["turns"]], m["duration"])
task_false = round(r["false"], 3)
print(task_false)

In [ ]:
expect('task_false', task_false, 2.054, tolerance=0.002)

> **Your turn.** (b) The cpWER of the far microphone with embeddings on IS1001a, with the recorded **Whisper base** transcript (`recorded["base"]`). Round to 3 decimals: `task_cpwer`.

In [ ]:
base = recorded["base"]
r = cpwer(ref_words("IS1001a"), setups(base, "IS1001a")["far_emb"])
task_cpwer = round(r["cpwer"], 3)
print(task_cpwer)

In [ ]:
expect('task_cpwer', task_cpwer, np.float64(0.774), tolerance=0.002)

> **Your turn.** (c) From `rows` of section 10, put the meeting and letter of the person with the highest WER (for example `"ES2004a.A"`) in `task_worst`.

In [ ]:
task_worst = max(rows)[2]
print(task_worst)

In [ ]:
expect_hash('task_worst', task_worst, '268d49f136883a19')

## Next

You measured who-spoke-when and the words of real meetings, compared a far microphone with headsets, and made notes with names. The [case study page](https://learning.nextia-ai.com/courses/speech/m10/meeting-transcripts-with-speakers/) has the recommendation, the costs, responsible use and how to make this a portfolio piece.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Spot commands on a small device](https://learning.nextia-ai.com/courses/speech/m10/spot-commands-on-a-small-device/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m10/meeting-transcripts-with-speakers/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The meetings are from the AMI Meeting Corpus (University of Edinburgh and the AMI Consortium, CC BY 4.0): real people in a role-play design meeting, recorded with their consent.